## Setup: Import libraries and configure environment

We'll use pandas for CSV data, numpy for arrays, and matplotlib for plotting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Environment configuration for MPI
Set the OpenMPI shared-memory transport backend to avoid issues with large process counts.


In [ ]:
%env OMPI_MCA_btl_vader_single_copy_mechanism=none
%env OMP_NUM_THREADS=1


In [ ]:
!chmod +x ../NDP_helper_scripts/*

In [ ]:
tandem_exc = 'tandem_2d_6p'
mpi_rank = 30
NDP_mpi_cmds = "--mca btl_vader_single_copy_mechanism none --mca btl_vader_backing_directory /tmp --bind-to none"
NDP_mpi_wrapper = "../NDP_helper_scripts/wrapper.sh"

## Run 1: Quasi-Dynamic (QD) mode — the standard approach

This run uses **mode = QD** in the <span style="color:orange">bp3_QD.toml</span>, which solves the fault stress balance quasi-statically at each time step. The solver inverts the elastic operator **K** fresh at each step—no precalculated Green's functions. We limit to 5 steps for demonstration speed. This is the simplest approach but slower because the linear system is solved repeatedly.


In [ ]:
! mpiexec  $NDP_mpi_cmds -n $mpi_rank  $NDP_mpi_wrapper $tandem_exc  bp3_QD.toml --petsc -ts_max_steps 10 -ts_monitor -options_file ../uniform/petsc_config.cfg

## Run 2: QD with Green's function precomputation (QDGreen, part 1)

This run uses **mode = QDGreen**, which precomputes and stores Green’s function operator **G** on the first stage. Subsequent steps reuse **G**, avoiding expensive linear solves. This is significantly faster for long simulations. Here, we will manually stop the run after a few Green’s function calculations for presentation purposes.

The GF are checkpointed to the temp dir by setting in the <span style="color:orange">bp3_QDGreen_1.toml</span>  file

```toml
[gf_checkpoint]
prefix = "temp"
```


In [ ]:
! mpiexec  $NDP_mpi_cmds -n $mpi_rank  $NDP_mpi_wrapper $tandem_exc  bp3_QDGreen_1.toml --petsc -options_file ../uniform/petsc_config.cfg

In [ ]:
!du -sh temp/* | sort -h

In [ ]:
!rm -r temp

In [ ]:
! ../NDP_helper_scripts/mpi_cleanup.sh

## Run 3: QD with Green's function — checkpoint and resume (QDGreen, part 2)
This second QDGreen run loads precalculated Green’s function from a checkpoint file. We will run it up to 7e9 seconds over the first earthquake cycle. The simulation will be underresolved for running time presentation purposes.

This will be done by setting in <span style="color:orange">bp3_QDGreen_2.toml</span>

```
final_time = 7e9
```
```
[gf_checkpoint]
prefix = "gf_res_0.50"
freq_cputime = 10
```

In [ ]:
! mpiexec  $NDP_mpi_cmds -n $mpi_rank  $NDP_mpi_wrapper $tandem_exc bp3_QDGreen_2.toml --petsc -ts_monitor -options_file ../uniform/petsc_config.cfg

## Analysis: Plot slip-rate evolution at a probe point on the fault

Read the fault probe output (15 km down-dip) and plot the magnitude of slip-rate versus time on a log scale. The time series reveals seismic ruptures (high slip-rate spikes) and low background rates across the simulated ~250-year cycle.


In [ ]:
sec_to_years = 356 * 24 * 3600
probe_15km = pd.read_csv('fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='15km')
plt.yscale('log')
plt.xlabel('Time (years)')
plt.ylabel('Slip rate (m/s)')

# Try On Your Own

Now that we understand the basics of BP3 simulations, try modifying the model parameters and running your own simulations. Remember to save your outputs to different prefixes so you don't overwrite the previuse results!

## Task 1: Change the Loading Rate

Modify the plate loading velocity and observe how it affects the timing of the first earthquake. 

**Steps:**
1. Open <span style="color:orange">bp3_QDGreen_2.toml</span>, change the prefix of the <span style="color:orange">fault_probe_output</span>
2. Find the <span style="color:orange">lib</span> parameter it is pointing to the <span style="color:orange">*.lua</span> note the <span style="color:orange">scenario</span> we are ran, open the <span style="color:orange">*.lua</span> file
3. Create a new `scenario` with double the plate rate (<span style="color:orange">Vp</span>) and change the <span style="color:orange">scenario</span> name in  `bp3_QDGreen_2.toml`
4. Run the simulation
5. Plot and compare the slip-rate time series with the original results


## Task 2: Change Frictional Properties

Modify the friction parameters of the fault and investigate how they affect rupture behavior.

**Steps:**
1. Change the <span style="color:orange">scenario</span> back to <span style="color:orange">scenario = "bp3_d60_normal"</span>
2. Edit the <span style="color:orange">*.lua</span> file
3. Adjust friction parameters or functions:
- <span style="color:orange">BP3.b</span> (rate-and-state friction evolution effect parameter)
- <span style="color:orange">function BP3:a</span> (rate-and-state friction direct effect parameter)
- <span style="color:orange">function BP3:L</span> (critical slip distance alsoe notated as $D_c$)
4. Run the simulation with a new output prefix

## Task 3: Vary Shear Modulus Between Subducting and Overriding Plates

Explore how differences in elastic properties between the two plates affect the system's behavior.

**Steps:**
1. Create a modified <span style="color:orange">*.lua</span> file where the subducting and overriding plates have different shear moduli (μ)
   - For example: `mu_overriding = 30 GPa`, `mu_subducting = 25 GPa` 
2. Update your <span style="color:orange">*.toml</span> file to use this modified <span style="color:orange">*.lua</span> file
3. **Recalculate the Green's functions** with changeing the `prfix` of `gf_checkpoint` pointing to a new directory (e.g., `gf_hetero`)
   - Green's functions depend on the elastic properties, so you must regenerate them
4. Run the full QDGreen simulation with the new GF checkpoint